In [1]:
import pandas as pd
from pathlib import Path

In [2]:
raw_data_path = "datasets/raw/merged_MIC_filtered20260312.csv"
df = pd.read_csv(raw_data_path)

/tmp/ipykernel_771124/3485414412.py:2: DtypeWarning: Columns (0: Source_ID, 1: Strain, 2: document_chembl_id, 3: assay_description, 4: molecule_chembl_id, 5: ASSAY_CELL_TYPE, 6: SRC_DESCRIPTION, 7: DOI, 8: DESCRIPTION, 9: COMPOUND_CODE, 10: COMPOUND_NAME, 11: PROJECT_ID, 12: LIBRARY_NAME, 13: ASSAY_ID, 14: activity_relation, 15: assay_tissue, 16: assay_cell_type, 17: ref_id, 18: ref_id_type, 19: target_id, 20: SourceID, 21: Target ChEMBL ID, 22: Accumulation phenotype, 23: Strain notes, 24: Phenotype, 25: Synonyms) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(raw_data_path)


In [3]:
print(df.iloc[0])

Source                                                               CHEMBL
Source_ID                                                          26105611
SMILES_std                CCCCCCCCC(=O)NCC(=O)N[C@@H](CC(C)C)C(=O)N[C@@H...
SMILES                    CCCCCCCCC(=O)NCC(=O)N[C@@H](CC(C)C)C(=O)N[C@@H...
ACTIVITY_TYPE                                                           MIC
ACTIVITY_VALUE                                                     5.060235
ACTIVITY_UNITS                                                      ug.mL-1
Strain                                                    Zymomonas mobilis
TARGET_NAME                                               Zymomonas mobilis
Genus                                                             Zymomonas
document_chembl_id                                            CHEMBL5608163
assay_description         Antibacterial activity against Zymomonas mobil...
molecule_chembl_id                                            CHEMBL5614208
ASSAY_CELL_T

In [4]:
print(df['Source'].unique())
print(df['Source_ID'].unique())
print(df['ACTIVITY_TYPE'].unique())
print(df['ACTIVITY_UNITS'].unique())
print(df['Strain'].unique())
print(df['TARGET_NAME'].unique())
print(df['Genus'].unique())

<ArrowStringArray>
['CHEMBL', 'Chemdiv', 'CO-ADD', 'NAPSS', 'Pubchem', 'SPARK']
Length: 6, dtype: str
[26105611 26105612 26105613 ... 'SPK-0000134' 'SPK-0000063' 'SPK-0000035']
<ArrowStringArray>
[       'MIC',      'MIC90',        'MBC',   'Activity',      'MIC99',
      'MIC80',     'MIC100',      'MBC90',      'MIC95',    'MBC99.9',
    'MIC=>80',     'MIC>80',     'MIC>90',    'MBC=>99',   'MBC99.99',
   'MBC>99.9',    'MIC=>95',    'MIC=>90',     'MIC>98',     'MBC100',
  'MBC=>99.9',      'MBC99',    'MIC99.9',     'MIC>95',     'MIC>85',
    'MIC=<90',     'MIC>99', 'Inhibition',     '#NAME?',      'MIC98']
Length: 30, dtype: str
<ArrowStringArray>
['ug.mL-1']
Length: 1, dtype: str
<ArrowStringArray>
[            'Zymomonas mobilis',                  'Yersinia sp.',
              'Yersinia ruckeri',   'Yersinia pseudotuberculosis',
               'Yersinia pestis',       'Yersinia enterocolitica',
           'Yarrowia lipolytica',       'Xanthomonas vesicatoria',
 'Xanthomonas o

In [5]:
usefull_columns = ["Source", "Source_ID", "SMILES_std", "ACTIVITY_VALUE", "Genus"]
df_usefull = df[usefull_columns]
df_usefull.rename(
    columns={
        "SMILES_std": "smiles",
        "ACTIVITY_VALUE": "value",
        "Genus": "genus",
        "Source": "source",
        "Source_ID": "source_id",
    },
    inplace=True,
)

In [6]:
print(f"original data size: {len(df_usefull)}")
df_usefull = df_usefull.drop_duplicates(subset=["smiles", "genus"])
print(f"after dropping duplicates: {len(df_usefull)}")

original data size: 1484205
after dropping duplicates: 379071


In [7]:
# give tags to the data
df_usefull["cid"] = df_usefull.index.map(lambda x: f"cid_{x:08d}")
df_usefull.to_csv("datasets/antibiotic_full_data.csv", index=False)

In [8]:
# 1. 统计每个 genus 的样本数
genus_counts = df_usefull["genus"].value_counts()

# 2. 取 top15 genus 名单
top15_genera = genus_counts.head(15).index.tolist()

# 3. 创建两个目录
top15_dir = Path("datasets/genus_top15")
other_dir = Path("datasets/genus_other")

top15_dir.mkdir(parents=True, exist_ok=True)
other_dir.mkdir(parents=True, exist_ok=True)

# 4. 按 genus 分开保存
for genus in df_usefull["genus"].dropna().unique():
    df_genus = df_usefull[df_usefull["genus"] == genus]
    
    if genus in top15_genera:
        save_path = top15_dir / f"{genus}.csv"
        group_name = "top15"
    else:
        save_path = other_dir / f"{genus}.csv"
        group_name = "other"
    
    print(f"[{group_name}] genus: {genus}, size: {len(df_genus)}")
    df_genus.to_csv(save_path, index=False)

# 5. 可选：把 top15 名单也存下来
genus_counts.to_csv("datasets/genus_counts.csv", header=["count"])

[other] genus: Zymomonas, size: 23
[top15] genus: Yersinia, size: 11498
[other] genus: Yarrowia, size: 50
[other] genus: Xanthomonas, size: 217
[other] genus: Wickerhamomyces, size: 66
[other] genus: Vibrio, size: 1291
[other] genus: Veillonella, size: 45
[other] genus: Ureaplasma, size: 15
[other] genus: Trueperella, size: 60
[other] genus: Trichosporon, size: 60
[other] genus: Trichophyton, size: 856
[other] genus: Trichoderma, size: 297
[other] genus: Treponema, size: 12
[other] genus: Syncephalastrum, size: 20
[other] genus: Stutzerimonas, size: 30
[other] genus: Streptomyces, size: 228
[top15] genus: Streptococcus, size: 17285
[other] genus: Stenotrophomonas, size: 1067
[top15] genus: Staphylococcus, size: 70059
[other] genus: Stachybotrys, size: 15
[other] genus: Sporothrix, size: 131
[other] genus: Spiroplasma, size: 10
[other] genus: Sphingomonas, size: 8
[top15] genus: Escherichia, size: 55281
[other] genus: Shigella, size: 2261
[other] genus: Shewanella, size: 21
[other] genu